# PicoCal - Transformer with pileup (notebook 10)

First look at minimum bias: train the transformer on `with_minimum_bias` vs `without_minimum_bias` and compare to BDT and LHCb `total_energy`. Data is small (~5 files each, pre-windowed ~9 cells) and has the selection confound from nb09, so this is exploratory. Small model, since Carla flagged the big one over-fits limited data.

In [1]:
import sys, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, split, resolution, TokenDS, collate, EPS

SEEDS = 5
cfg = {"d": 48, "nhead": 4, "layers": 2, "dropout": 0.2, "lr": 5e-4, "wd": 1e-3,
       "batch": 64, "epochs": 200, "patience": 25}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
with_files = sorted((repo / "data" / "gsoc_drive" / "with_minimum_bias").glob("matched_*.root"))
without_files = sorted((repo / "data" / "gsoc_drive" / "without_minimum_bias").glob("matched_*.root"))
{"device": DEVICE, "with_files": len(with_files), "without_files": len(without_files)}

{'device': 'cuda', 'with_files': 5, 'without_files': 5}

In [2]:
class TunedTransformer(nn.Module):
    def __init__(self, in_dim, d=48, nhead=4, layers=2, dropout=0.2):
        super().__init__()
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=4 * d, dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))

    def forward(self, x, m):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = m.unsqueeze(-1).float()
        return self.head((h * w).sum(1) / w.sum(1).clamp(min=1))


def train_one(make_model, tr, toks, y, va, te, Et, seed, verbose=False):
    torch.manual_seed(seed)
    cont = np.concatenate([toks[i][:, :7] for i in tr], 0)
    mean = cont.mean(0); std = cont.std(0) + EPS

    def loader(idx, sh):
        return DataLoader(TokenDS([toks[i] for i in idx], y[idx], mean, std, 7),
                          batch_size=cfg["batch"], shuffle=sh, collate_fn=collate)

    model = make_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    dl_tr, dl_va = loader(tr, True), loader(va, False)

    def vloss():
        model.eval(); t = 0.0; n = 0
        with torch.no_grad():
            for X, m, yb in dl_va:
                t += nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE)).item(); n += 1
        return t / max(n, 1)

    hist = {"epoch": [], "train_loss": [], "val_loss": []}
    best = float("inf"); best_state = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train(); tl = 0.0; nb = 0
        for X, m, yb in dl_tr:
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE))
            loss.backward(); opt.step(); tl += loss.item(); nb += 1
        sched.step(); v = vloss(); tr_l = tl / max(nb, 1)
        hist["epoch"].append(ep); hist["train_loss"].append(tr_l); hist["val_loss"].append(v)
        if verbose:
            print(f"  epoch {ep:3d}  train {tr_l:.4f}  val {v:.4f}", flush=True)
        if v < best - 1e-4:
            best = v; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(best_state)

    def predict(idx):
        model.eval(); out = []
        with torch.no_grad():
            for X, m, _ in loader(idx, False):
                out.append(model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel())
        return np.concatenate(out)

    pv, pt = predict(va), predict(te)
    a, b = np.polyfit(pv, y[va], 1)
    return resolution(np.exp(a * pt + b), Et[te])["sigma_eff"], hist

In [3]:
def build_ds(files):
    return build(files, 3, 100.0, selector=lambda c: np.ones(len(c["energy"]), dtype=bool))

def evaluate(D, label, keep_hist=False):
    n = len(D["y"]); tr, va, te = split(n)
    toks = D["tok_seed"]; y = D["y"]; Et = D["Etrue"]
    in_dim = toks[0].shape[1]
    mk = lambda: TunedTransformer(in_dim, cfg["d"], cfg["nhead"], cfg["layers"], cfg["dropout"])
    vals = []; hist0 = None
    for s in range(SEEDS):
        sig, h = train_one(mk, tr, toks, y, va, te, Et, s, verbose=(keep_hist and s == 0))
        vals.append(sig)
        if s == 0:
            hist0 = h
    gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(D["agg"][tr], y[tr])
    bdt = resolution(np.exp(gb.predict(D["agg"][te])), Et[te])["sigma_eff"]
    a, b = np.polyfit(np.log(D["total_energy"][tr] + EPS), y[tr], 1)
    teg = resolution(np.exp(a * np.log(D["total_energy"][te] + EPS) + b), Et[te])["sigma_eff"]
    return {"data": label, "n": int(n), "median_cells": int(np.median([toks[i].shape[0] for i in range(n)])),
            "transformer_mean": round(float(np.mean(vals)), 4), "transformer_std": round(float(np.std(vals)), 4),
            "BDT": round(float(bdt), 4), "total_energy": round(float(teg), 4)}, hist0

Dwithout = build_ds(without_files)
Dwith = build_ds(with_files)
r_without, _ = evaluate(Dwithout, "without_pileup")
r_with, hist_with = evaluate(Dwith, "with_pileup", keep_hist=True)
res = [r_without, r_with]
res

  epoch   0  train 7.6046  val 4.8075


  epoch   1  train 3.3582  val 2.2207


  epoch   2  train 1.6661  val 1.1488


  epoch   3  train 0.9766  val 0.9714


  epoch   4  train 0.9197  val 0.7877


...

  epoch 114  train 0.1598  val 0.0677


  epoch 115  train 0.1698  val 0.0514


  epoch 116  train 0.1572  val 0.0389


  epoch 117  train 0.1529  val 0.0616


[{'data': 'without_pileup',
  'n': 1837,
  'median_cells': 9,
  'transformer_mean': 0.4402,
  'transformer_std': 0.021,
  'BDT': 0.4478,
  'total_energy': 0.4892},
 {'data': 'with_pileup',
  'n': 893,
  'median_cells': 9,
  'transformer_mean': 0.169,
  'transformer_std': 0.0059,
  'BDT': 0.1636,
  'total_energy': 0.1031}]

In [4]:
pd.DataFrame(res)[["data", "n", "median_cells", "transformer_mean", "transformer_std", "BDT", "total_energy"]]

,data,n,median_cells,transformer_mean,transformer_std,BDT,total_energy
0,without_pileup,1837,9,0.4402,0.0210,0.4478,0.4892
1,with_pileup,893,9,0.1690,0.0059,0.1636,0.1031


In [5]:
import plotly.graph_objects as go
cats = ["transformer", "BDT", "total_energy"]
fig = go.Figure()
for r, col in [(r_without, "#4c78a8"), (r_with, "#e45756")]:
    fig.add_trace(go.Bar(name=r["data"], x=cats,
                         y=[r["transformer_mean"], r["BDT"], r["total_energy"]],
                         error_y=dict(type="data", array=[r["transformer_std"], 0, 0]),
                         marker_color=col,
                         text=[f'{r["transformer_mean"]:.3f}', f'{r["BDT"]:.3f}', f'{r["total_energy"]:.3f}'],
                         textposition="outside"))
fig.update_layout(barmode="group", template="plotly_white", height=440,
                  title="Energy resolution: with vs without pileup (lower is better)",
                  yaxis_title="sigma_eff", legend_title="")
fig.show()

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=hist_with["epoch"], y=hist_with["train_loss"], mode="lines",
                         name="train", line=dict(color="#4c78a8")))
fig.add_trace(go.Scatter(x=hist_with["epoch"], y=hist_with["val_loss"], mode="lines",
                         name="val", line=dict(color="#e45756", dash="dash")))
fig.update_layout(template="plotly_white", height=400, title="Learning curve (with pileup, seed 0)",
                  xaxis_title="epoch", yaxis_title="MSE loss (log-energy)", legend_title="")
fig.show()

Read the bar chart by the **margin**: on clean signal the transformer barely beats BDT / `total_energy` (the sum-dominated regime). If the transformer's edge over the rule-based estimators is **larger with pileup than without**, that is the first hint the attention model earns its keep once background is present. Caveat: tiny sample (~5 files), pre-windowed cells, and the nb09 selection confound - treat as a direction, not a number. Next: more files from Felipe, per-event with/without matching, and feature-importance under pileup.